In [ ]:
import os

os.chdir("spaceship-titanic")

In [ ]:
with open(".gitignore", "w") as f:
  f.write("# Files to ignore\n")

In [ ]:
with open(".gitignore", "a") as f:
  f.write("train.csv\ntest.csv\n")

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.pipeline import Pipeline, make_pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split, GridSearchCV

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv


In [ ]:
import io
import pandas as pd

# Get the list of uploaded filenames
file_names = list(uploaded.keys())

# Load the files
test_df = pd.read_csv(io.BytesIO(uploaded[file_names[0]]))
train_df = pd.read_csv(io.BytesIO(uploaded[file_names[1]]))

In [ ]:
# No of rows in train data
train_len = len(train_df)

# Concatenate train and test data
df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

In [ ]:
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12970 entries, 0 to 12969
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  object 
 1   HomePlanet    12682 non-null  object 
 2   CryoSleep     12660 non-null  object 
 3   Cabin         12671 non-null  object 
 4   Destination   12696 non-null  object 
 5   Age           12700 non-null  float64
 6   VIP           12674 non-null  object 
 7   RoomService   12707 non-null  float64
 8   FoodCourt     12681 non-null  float64
 9   ShoppingMall  12664 non-null  float64
 10  Spa           12686 non-null  float64
 11  VRDeck        12702 non-null  float64
 12  Name          12676 non-null  object 
 13  Transported   8693 non-null   object 
dtypes: float64(6), object(8)
memory usage: 1.4+ MB


In [ ]:
df.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,12700.000000,12707.000000,12681.000000,12664.000000,12686.000000,12702.000000
mean,28.771969,222.897852,451.961675,174.906033,308.476904,306.789482
std,14.387261,647.596664,1584.370747,590.558690,1130.279641,1180.097223
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,49.000000,77.000000,29.000000,57.000000,42.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [ ]:
df.describe(include=['O'])

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,VIP,Name,Transported
count,12970,12682,12660,12671,12696,12674,12676,8693
unique,12970,3,2,9825,3,2,12629,2
top,9277_01,Earth,False,G/734/S,TRAPPIST-1e,False,Artis Gambs,True
freq,1,6865,8079,8,8871,12401,2,4378


In [ ]:
df.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'Transported'],
      dtype='object')

In [ ]:
df["PassengerId"]

,PassengerId
0,0001_01
1,0002_01
2,0003_01
3,0003_02
4,0004_01
...,...
12965,9266_02
12966,9269_01
12967,9271_01
12968,9273_01


In [ ]:
# Extract "GroupNo" from PassengerId column
df["GroupNo"] = df["PassengerId"].str.split(pat="_", expand=True)[0].astype(int)

In [ ]:
df["GroupNo"]

,GroupNo
0,1
1,2
2,3
3,3
4,4
...,...
12965,9266
12966,9269
12967,9271
12968,9273


In [ ]:
# Find the number of members in the specific passenger's group
df["GroupSize"] = df.groupby(["GroupNo"])["GroupNo"].transform("count")

In [ ]:
df["GroupSize"]

,GroupSize
0,1
1,1
2,2
3,2
4,1
...,...
12965,2
12966,1
12967,1
12968,1


In [ ]:
df.groupby(["GroupSize"], as_index=False)["Transported"].mean()

,GroupSize,Transported
0,1,0.452445
1,2,0.53805
2,3,0.593137
3,4,0.640777
4,5,0.592453
5,6,0.614943
6,7,0.541126
7,8,0.394231


In [ ]:
# Divide group size into "Alone", "Small", "Medium" and "Large"
group_map = {1: "Alone", 2: "Small", 3: "Small", 4:"Small", 5: "Medium", 6: "Medium", 7: "Large", 8: "Large"}

df["GroupSize_grouped"] = df["GroupSize"].map(group_map)

In [ ]:
df.groupby(["GroupSize_grouped"], as_index=False)["Transported"].mean()

,GroupSize_grouped,Transported
0,Alone,0.452445
1,Large,0.495522
2,Medium,0.601367
3,Small,0.569685


In [ ]:
df.groupby(["HomePlanet"], as_index=False)["Transported"].mean()

,HomePlanet,Transported
0,Earth,0.423946
1,Europa,0.658846
2,Mars,0.523024


In [ ]:
df.groupby(["CryoSleep"], as_index=False)["Transported"].mean()

,CryoSleep,Transported
0,False,0.328921
1,True,0.817583


In [ ]:
df["CryoSleep"] = df["CryoSleep"].astype(float)

In [ ]:
df.groupby(["Destination"], as_index=False)["Transported"].mean()

,Destination,Transported
0,55 Cancri e,0.61
1,PSO J318.5-22,0.503769
2,TRAPPIST-1e,0.471175


In [55]:
df["Cabin"].unique() #deck/num/size

array(['B/0/P', 'F/0/S', 'A/0/S', ..., 'D/296/P', 'D/297/P', 'G/1498/S'],
      dtype=object)

In [56]:
# Extract "Deck" from Cabin column
df["Deck"] = df["Cabin"].str.split(pat="/", expand=True)[0]

In [57]:
df["Deck"]

,Deck
0,B
1,F
2,A
3,A
4,F
...,...
12965,G
12966,NaN
12967,D
12968,D


In [58]:
df.groupby(["Deck"], as_index=False)["Transported"].mean()

,Deck,Transported
0,A,0.496094
1,B,0.734275
2,C,0.680054
3,D,0.433054
4,E,0.357306
5,F,0.439871
6,G,0.516217
7,T,0.2


In [59]:
# Extract "CabinNo" from Cabin column
df["CabinNo"] = df["Cabin"].str.split(pat="/", expand=True)[1]

In [60]:
df["CabinNo"]

,CabinNo
0,0
1,0
2,0
3,0
4,1
...,...
12965,1496
12966,NaN
12967,296
12968,297


In [61]:
df["CabinNo"] = pd.to_numeric(df["CabinNo"], errors="coerce")

In [62]:
# Extract "CabinSize" from Cabin column
df["CabinSize"] = df["Cabin"].str.split(pat="/", expand=True)[2]

In [63]:
df["CabinSize"]

,CabinSize
0,P
1,S
2,S
3,S
4,S
...,...
12965,S
12966,NaN
12967,P
12968,P


In [64]:
df.groupby(["CabinSize"], as_index=False)["Transported"].mean()

,CabinSize,Transported
0,P,0.45126
1,S,0.555037


In [66]:
df.groupby(["VIP"], as_index=False)["Transported"].mean()

,VIP,Transported
0,False,0.506332
1,True,0.38191


In [67]:
df["VIP"] = df["VIP"].astype(float)

In [68]:
# Create "MoneySpent" by adding all the expenses made by the passenger on RoomService, FoodCourt, ShoppingMall, Spa, VRDeck
df["MoneySpent"] = df["RoomService"] + df["FoodCourt"] + df["ShoppingMall"] + df["Spa"] + df["VRDeck"]

In [69]:
df["MoneySpent"]

,MoneySpent
0,0.0
1,736.0
2,10383.0
3,5176.0
4,1091.0
...,...
12965,0.0
12966,1018.0
12967,0.0
12968,3203.0
